# Alpha158 실전 분석

이 노트북은 이 저장소가 생성한 **실제 Qlib binary provider**를 읽어 Alpha158을 진단합니다. 데이터 다운로드나 합성 데이터 대체는 하지 않습니다.

분석 순서: export 계약 확인 → 158개 RAW 특징 계산 → 결측·분포·상관 분석 → `D+1 open` 진입, `D+5 close` 청산의 SPY 초과수익 라벨 → IC/Rank IC → 분위수 분석 → 최신일 탐색 순위.

> 주의: 마지막 순위는 단일 특징의 탐색 결과이며 운영 모델 점수나 투자 권유가 아닙니다. 랜덤 분할, 전체 기간 정규화, 미래 라벨을 특징에 섞는 방식은 금지합니다.

## 0. 실행 환경과 사용자 설정

- Python 3.11 또는 3.12와 `pip install -e ".[analytics]"` 환경을 권장합니다.
- `PROVIDER_URI`가 `None`이면 `data/output/qlib/*`에서 가장 최근 수정된 유효 export를 선택합니다.
- 날짜가 `None`이면 provider calendar 전체를 사용합니다. Alpha158은 최대 60 session lookback을 사용하므로 시작부 결측은 정상일 수 있습니다.

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROVIDER_URI: str | Path | None = None
START_DATE: str | None = None
END_DATE: str | None = None
INSTRUMENTS = "all"
BENCHMARK = "SPY"
MIN_CROSS_SECTION = 30
TOP_N = 10
FEATURE_TO_ANALYZE: str | None = None

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("pyproject.toml이 있는 저장소 루트를 찾지 못했습니다.")

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

REPO_ROOT


In [ ]:
def discover_export(root: Path, override: str | Path | None) -> tuple[Path, Path]:
    if override is not None:
        provider = Path(override).expanduser().resolve()
        export_dir = provider.parent if provider.name == "qlib_data" else provider
        provider = provider if provider.name == "qlib_data" else provider / "qlib_data"
        candidates = [export_dir]
    else:
        output = root / "data" / "output" / "qlib"
        candidates = sorted(
            (path for path in output.glob("*") if path.is_dir()),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    for export_dir in candidates:
        provider = export_dir / "qlib_data"
        if (export_dir / "manifest.json").is_file() and provider.is_dir():
            return export_dir, provider
    raise FileNotFoundError(
        "유효한 Qlib export를 찾지 못했습니다. data/output/qlib/<run-id>/ 아래에 "
        "manifest.json과 qlib_data가 있는지 확인하거나 PROVIDER_URI를 지정하세요."
    )

EXPORT_DIR, PROVIDER_PATH = discover_export(REPO_ROOT, PROVIDER_URI)
manifest = json.loads((EXPORT_DIR / "manifest.json").read_text(encoding="utf-8"))
feature_config = yaml.safe_load(
    (REPO_ROOT / "config" / "analytics" / "feature_alpha158_v1.yaml").read_text(encoding="utf-8")
)
label_config = yaml.safe_load(
    (REPO_ROOT / "config" / "analytics" / "label_excess_return_5d_v1.yaml").read_text(encoding="utf-8")
)
calendar = pd.DatetimeIndex(
    pd.read_csv(PROVIDER_PATH / "calendars" / "day.txt", header=None)[0]
).sort_values()
instrument_table = pd.read_csv(
    PROVIDER_PATH / "instruments" / "all.txt",
    sep="\t",
    header=None,
    names=["instrument", "start", "end"],
)
analysis_start = pd.Timestamp(START_DATE) if START_DATE else calendar.min()
analysis_end = pd.Timestamp(END_DATE) if END_DATE else calendar.max()
if analysis_start > analysis_end:
    raise ValueError("START_DATE는 END_DATE보다 늦을 수 없습니다.")
if analysis_start < calendar.min() or analysis_end > calendar.max():
    raise ValueError(f"분석 기간은 provider calendar 범위 {calendar.min().date()}~{calendar.max().date()} 안이어야 합니다.")

display(pd.Series(manifest, name="value").to_frame())
display(pd.DataFrame({
    "항목": ["provider", "calendar", "분석 기간", "종목 수", "Alpha158 차원", "lookback", "라벨"],
    "값": [
        str(PROVIDER_PATH),
        f"{calendar.min().date()} ~ {calendar.max().date()} ({len(calendar)} sessions)",
        f"{analysis_start.date()} ~ {analysis_end.date()}",
        len(instrument_table),
        feature_config["expected_dimension"],
        feature_config["lookback_sessions"],
        label_config["name"],
    ],
}))
display(instrument_table.head())


## 1. Alpha158 RAW 특징 계산과 계약 검증

저장소의 `Alpha158FeatureService`를 그대로 사용합니다. 설정상 infer/learn processor가 비어 있으므로 이 단계는 정규화 전 RAW 특징입니다. 정확히 158차원이어야 하고, 중복 index와 무한대는 허용하지 않습니다.

In [ ]:
from sp500_candidate_engine.analytics.features import Alpha158FeatureService

result = Alpha158FeatureService().compute_raw(
    provider_uri=PROVIDER_PATH,
    instruments=INSTRUMENTS,
    start_date=analysis_start.date(),
    end_date=analysis_end.date(),
)
raw_features = result.frame.copy()

def normalize_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(frame.index, pd.MultiIndex) or frame.index.nlevels != 2:
        raise ValueError(f"Alpha158 index는 2-level MultiIndex여야 합니다: {frame.index}")
    index_frame = frame.index.to_frame(index=False)
    datetime_column = next(
        (column for column in index_frame if str(column).lower() in {"datetime", "date"}),
        None,
    )
    if datetime_column is None:
        scores = {column: pd.to_datetime(index_frame[column], errors="coerce").notna().mean() for column in index_frame}
        datetime_column = max(scores, key=scores.get)
        if scores[datetime_column] < 0.95:
            raise ValueError("Alpha158 index에서 datetime level을 식별하지 못했습니다.")
    instrument_column = next(column for column in index_frame if column != datetime_column)
    flat = frame.reset_index()
    flat = flat.rename(columns={datetime_column: "datetime", instrument_column: "instrument"})
    flat["datetime"] = pd.to_datetime(flat["datetime"])
    flat["instrument"] = flat["instrument"].astype(str).str.upper()
    feature_columns = [column for column in flat.columns if column not in {"datetime", "instrument"}]
    flat = flat.set_index(["datetime", "instrument"])[feature_columns].sort_index()
    flat.columns = [str(column[-1] if isinstance(column, tuple) else column) for column in flat.columns]
    return flat

features = normalize_feature_frame(raw_features)
assert features.shape[1] == 158, features.shape
assert not features.index.has_duplicates
assert not np.isinf(features.to_numpy(dtype=float)).any()

print(f"shape={features.shape}, dates={features.index.get_level_values('datetime').nunique()}, "
      f"instruments={features.index.get_level_values('instrument').nunique()}")
display(features.head())


## 2. 특징 이름과 결측 분석

아래 분류는 Qlib 표현식 이름을 이해하기 위한 **휴리스틱**이며 공식 taxonomy가 아닙니다. 시작부 결측은 rolling lookback(최대 60 sessions)의 정상적인 warm-up일 수 있지만, 후반부의 갑작스러운 coverage 하락은 데이터 품질 문제일 수 있습니다.

In [ ]:
def feature_group(name: str) -> str:
    upper = name.upper()
    if any(token in upper for token in ("CORR", "CORD", "RSQR", "RESI", "BETA")):
        return "correlation/regression"
    if any(token in upper for token in ("VOLUME", "VMA", "VSTD", "VSUMP", "VSUMN")):
        return "volume"
    if any(token in upper for token in ("KMID", "KLEN", "KUP", "KLOW", "KSFT")):
        return "K-bar/candlestick"
    if any(token in upper for token in ("ROC", "MA", "STD", "MAX", "MIN", "QTL", "RANK", "SUM")):
        return "rolling statistics"
    if any(token in upper for token in ("OPEN", "HIGH", "LOW", "CLOSE", "VWAP")):
        return "price"
    return "other"

feature_catalog = pd.DataFrame({"feature": features.columns})
feature_catalog["heuristic_group"] = feature_catalog["feature"].map(feature_group)
display(feature_catalog.groupby("heuristic_group").size().rename("count").sort_values(ascending=False).to_frame())
display(feature_catalog.head(30))

missing_by_feature = features.isna().mean().sort_values(ascending=False)
missing_by_date = features.isna().mean(axis=1).groupby(level="datetime").mean()
missing_by_instrument = features.isna().mean(axis=1).groupby(level="instrument").mean().sort_values(ascending=False)

display(pd.DataFrame({
    "전체 결측률": [features.isna().to_numpy().mean()],
    "완전 결측 특징 수": [(missing_by_feature == 1).sum()],
    "완전 관측 특징 수": [(missing_by_feature == 0).sum()],
}))
display(missing_by_feature.head(20).rename("missing_rate").to_frame())
display(missing_by_instrument.head(20).rename("missing_rate").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
missing_by_feature.head(20).sort_values().plot.barh(ax=axes[0], title="결측률 상위 20개 특징")
(1 - missing_by_date).plot(ax=axes[1], title="일별 평균 feature coverage")
axes[0].set_xlabel("missing rate")
axes[1].set_ylabel("coverage")
plt.tight_layout()


## 3. 분포·시간 안정성·상관

시각화에서만 1~99 percentile로 clipping합니다. 원본 특징은 수정하지 않습니다. 상관 heatmap은 결측률이 낮은 특징 중 최대 24개를 표본으로 사용하며, 높은 상관 쌍을 자동 제거하지 않습니다.

In [ ]:
selected = list(missing_by_feature.sort_values().head(6).index)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for feature, axis in zip(selected, axes.flat, strict=True):
    values = features[feature].dropna()
    low, high = values.quantile([0.01, 0.99])
    values.clip(low, high).hist(bins=50, ax=axis)
    axis.set_title(feature, fontsize=9)
plt.suptitle("선택 특징 분포 (시각화만 1~99% clipping)")
plt.tight_layout()

daily = features[selected].groupby(level="datetime")
medians = daily.median()
q25 = daily.quantile(0.25)
q75 = daily.quantile(0.75)
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
for feature, axis in zip(selected, axes.flat, strict=True):
    axis.plot(medians.index, medians[feature], label="median")
    axis.fill_between(medians.index, q25[feature], q75[feature], alpha=0.25, label="IQR")
    axis.set_title(feature, fontsize=9)
plt.tight_layout()

correlation_features = list(missing_by_feature.sort_values().head(24).index)
sample = features[correlation_features]
if len(sample) > 50_000:
    sample = sample.sample(50_000, random_state=42)
corr = sample.corr(method="spearman", min_periods=MIN_CROSS_SECTION)
fig, axis = plt.subplots(figsize=(11, 9))
image = axis.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
axis.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
axis.set_yticks(range(len(corr)), corr.index, fontsize=7)
fig.colorbar(image, ax=axis, label="Spearman correlation")
axis.set_title("표본 특징 상관 heatmap")
plt.tight_layout()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
high_pairs = upper[upper.abs() >= 0.90].sort_values(key=lambda series: series.abs(), ascending=False)
display(high_pairs.rename("spearman_corr").head(30).to_frame())


## 4. 실제 가격에서 5-session SPY 초과수익 라벨 생성

anchor가 D일 때 entry는 **다음 거래 session(D+1)의 adjusted open**, exit는 **D+5 거래 session의 adjusted close**입니다. 같은 entry/exit의 SPY 수익률을 빼며 calendar-day shift를 사용하지 않습니다.

In [ ]:
staging_files = sorted((EXPORT_DIR / "staging").glob("*.csv"))
if not staging_files:
    raise FileNotFoundError(f"가격 staging CSV가 없습니다: {EXPORT_DIR / 'staging'}")

prices = pd.concat(
    (pd.read_csv(path, usecols=["date", "symbol", "open", "high", "low", "close", "volume"]) for path in staging_files),
    ignore_index=True,
)
prices["date"] = pd.to_datetime(prices["date"])
prices["symbol"] = prices["symbol"].astype(str).str.upper()
prices = prices.sort_values(["symbol", "date"])
if prices.duplicated(["symbol", "date"]).any():
    raise ValueError("staging 가격에 symbol/date 중복이 있습니다.")
if BENCHMARK.upper() not in set(prices["symbol"]):
    raise ValueError(f"benchmark {BENCHMARK!r}가 export에 없습니다. BENCHMARK 또는 export universe를 확인하세요.")

calendar_series = pd.Series(calendar, index=calendar)
entry_dates = calendar_series.shift(-1)
exit_dates = calendar_series.shift(-5)
indexed_prices = prices.set_index(["symbol", "date"])
benchmark = indexed_prices.loc[BENCHMARK.upper()].reindex(calendar)
benchmark_return = benchmark["close"].shift(-5) / benchmark["open"].shift(-1) - 1

label_parts: list[pd.DataFrame] = []
for symbol in prices["symbol"].unique():
    symbol_prices = indexed_prices.loc[symbol].reindex(calendar)
    raw_return = symbol_prices["close"].shift(-5) / symbol_prices["open"].shift(-1) - 1
    label_parts.append(pd.DataFrame({
        "datetime": calendar,
        "instrument": symbol,
        "entry_date": entry_dates.to_numpy(),
        "exit_date": exit_dates.to_numpy(),
        "raw_return": raw_return.to_numpy(),
        "benchmark_return": benchmark_return.to_numpy(),
        "label": (raw_return - benchmark_return).to_numpy(),
    }))
labels = pd.concat(label_parts, ignore_index=True).set_index(["datetime", "instrument"]).sort_index()
label_coverage = labels["label"].notna().groupby(level="datetime").mean()
print(f"price rows={len(prices):,}, label rows={len(labels):,}, overall label coverage={labels['label'].notna().mean():.2%}")
display(labels.dropna(subset=["label"]).head())
label_coverage.plot(figsize=(13, 3), title="일별 5-session excess-return label coverage", ylabel="coverage")
plt.tight_layout()


## 5. 특징-라벨 결합과 IC / Rank IC

Pearson IC와 Spearman Rank IC는 날짜별 횡단면에서 계산합니다. 유효 표본이 `MIN_CROSS_SECTION` 미만이거나 특징/라벨이 상수이면 NaN을 반환합니다. 여기서 ICIR은 `일별 IC 평균 / 일별 IC 표준편차`의 비연율화 convention입니다.

In [ ]:
analysis = features.join(labels, how="inner")
valid_dates = analysis.index.get_level_values("datetime")
if not ((analysis["entry_date"].isna()) | (analysis["entry_date"] > valid_dates)).all():
    raise AssertionError("entry_date는 feature anchor보다 뒤여야 합니다.")
if not ((analysis["exit_date"].isna()) | (analysis["exit_date"] > valid_dates)).all():
    raise AssertionError("exit_date는 feature anchor보다 뒤여야 합니다.")
feature_columns = list(features.columns)
print(f"merged rows={len(analysis):,}, valid labels={analysis['label'].notna().sum():,}")

def safe_corr(frame: pd.DataFrame, feature: str, method: str) -> float:
    pair = frame[[feature, "label"]].dropna()
    if len(pair) < MIN_CROSS_SECTION or pair[feature].nunique() < 2 or pair["label"].nunique() < 2:
        return np.nan
    return float(pair[feature].corr(pair["label"], method=method))

pearson_rows: dict[pd.Timestamp, dict[str, float]] = {}
rank_rows: dict[pd.Timestamp, dict[str, float]] = {}
for date_value, cross_section in analysis.groupby(level="datetime", sort=True):
    pearson_rows[date_value] = {feature: safe_corr(cross_section, feature, "pearson") for feature in feature_columns}
    rank_rows[date_value] = {feature: safe_corr(cross_section, feature, "spearman") for feature in feature_columns}

daily_ic = pd.DataFrame.from_dict(pearson_rows, orient="index").sort_index()
daily_rank_ic = pd.DataFrame.from_dict(rank_rows, orient="index").sort_index()

def summarize_ic(values: pd.DataFrame, prefix: str) -> pd.DataFrame:
    mean = values.mean()
    std = values.std(ddof=1)
    return pd.DataFrame({
        f"{prefix}_mean": mean,
        f"{prefix}_std": std,
        f"{prefix}_ICIR": mean / std.replace(0, np.nan),
        f"{prefix}_positive_ratio": values.gt(0).sum() / values.notna().sum().replace(0, np.nan),
        f"{prefix}_observations": values.notna().sum(),
    })

ic_summary = summarize_ic(daily_ic, "IC").join(summarize_ic(daily_rank_ic, "RankIC"))
ic_summary = ic_summary.sort_values("RankIC_mean", ascending=False)
display(ic_summary.head(20))
display(ic_summary.tail(20).sort_values("RankIC_mean"))

best_features = list(ic_summary.head(5).index)
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
daily_rank_ic[best_features].rolling(20, min_periods=10).mean().plot(ax=axes[0], title="상위 특징 20-session rolling Rank IC")
daily_ic[best_features].rolling(20, min_periods=10).mean().plot(ax=axes[1], title="상위 특징 20-session rolling Pearson IC")
for axis in axes:
    axis.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()


## 6. 단일 특징 분위수 분석

기본적으로 평균 Rank IC가 가장 높은 특징을 사용합니다. `FEATURE_TO_ANALYZE`에 정확한 특징 이름을 지정해 바꿀 수 있습니다. 날짜별 percentile rank 후 5분위로 나누므로 중복 값에도 안전합니다.

In [ ]:
chosen_feature = FEATURE_TO_ANALYZE or str(ic_summary["RankIC_mean"].idxmax())
if chosen_feature not in feature_columns:
    raise KeyError(f"FEATURE_TO_ANALYZE={chosen_feature!r}가 Alpha158 특징에 없습니다.")

quintile_input = analysis[[chosen_feature, "label"]].dropna().copy()
quintile_input["percentile"] = quintile_input.groupby(level="datetime")[chosen_feature].rank(method="average", pct=True)
quintile_input["quintile"] = np.minimum(np.ceil(quintile_input["percentile"] * 5), 5).astype(int)
daily_quintile = quintile_input.groupby([pd.Grouper(level="datetime"), "quintile"])["label"].mean().unstack()
quintile_mean = daily_quintile.mean().rename(index=lambda value: f"Q{int(value)}")
spread = daily_quintile.get(5, pd.Series(dtype=float)) - daily_quintile.get(1, pd.Series(dtype=float))

print(f"chosen feature: {chosen_feature}")
display(quintile_mean.rename("mean_excess_return").to_frame())
print(f"Q5-Q1 mean spread={spread.mean():.4%}, positive ratio={(spread > 0).mean():.2%}, observations={spread.notna().sum()}")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
quintile_mean.plot.bar(ax=axes[0], title="평균 5-session 초과수익 by quintile", ylabel="excess return")
spread.rolling(20, min_periods=10).mean().plot(ax=axes[1], title="Q5-Q1 spread 20-session rolling mean")
axes[1].axhline(0, color="black", linewidth=0.8)
plt.tight_layout()


## 7. 최신일 탐색 순위

라벨이 아니라 최신 anchor에서 관측 가능한 단일 특징만 사용합니다. 방향은 평균 Rank IC의 부호로 맞춥니다. 이는 **탐색용 순위**이며, LightGBM score나 운영 후보 선정 결과가 아닙니다.

In [ ]:
latest_date = features[chosen_feature].dropna().index.get_level_values("datetime").max()
latest = features.xs(latest_date, level="datetime")[[chosen_feature]].dropna().copy()
direction = 1.0 if ic_summary.loc[chosen_feature, "RankIC_mean"] >= 0 else -1.0
latest["directed_score"] = latest[chosen_feature] * direction
latest["percentile"] = latest["directed_score"].rank(method="average", pct=True)
latest = latest.sort_values(["percentile", "directed_score"], ascending=False).head(TOP_N)

latest_prices = prices.loc[prices["date"].eq(latest_date), ["symbol", "open", "high", "low", "close", "volume"]].set_index("symbol")
latest = latest.join(latest_prices, how="left")
print(f"latest anchor={latest_date.date()}, feature={chosen_feature}, direction={'positive' if direction > 0 else 'negative'}")
display(latest)


## 8. 누수 감사와 결과 해석 체크리스트

이 노트북이 확인한 사항:

- 특징 index는 anchor D이고 entry/exit는 각각 D보다 뒤입니다.
- 라벨 shift는 provider 거래 calendar 기준이며 calendar day가 아닙니다.
- RAW Alpha158은 processor 없이 계산했고 Infinity와 index 중복을 차단했습니다.
- 미래 가격을 제거해도 과거 특징이 바뀌지 않는 강한 검증은 provider 복제가 필요하므로 노트북에서 흉내 내지 않습니다. 저장소의 Qlib runtime/integration test와 별도의 truncation test로 검증해야 합니다.

운영 모델로 확장할 때:

1. 랜덤 split 대신 시간 순서 train/validation/test 또는 walk-forward를 사용합니다.
2. 경계에서 exit가 segment를 넘는 표본은 최소 5-session purge합니다.
3. 표준화·winsorization 등 processor는 train 구간에서만 fit하고 validation/test에는 저장 artifact로 transform만 합니다.
4. 평균 Rank IC뿐 아니라 fold별 안정성, ICIR, coverage, sector concentration, Q5-Q1 spread를 함께 봅니다.
5. 1년 데이터 결과는 개발/shadow 근거이며, 3~5년 point-in-time universe에서 LightGBM과 momentum/Ridge baseline을 walk-forward 비교합니다.

### 결과 기록 템플릿

- 분석 export / 기간 / 종목 수:
- 전체 결측률과 비정상 coverage 구간:
- 평균 Rank IC 상·하위 특징과 안정성:
- 선택 특징 Q5-Q1 spread와 양수 비율:
- 높은 상관 특징군:
- 데이터·survivorship·corporate-action 제한:
- 다음 실험: chronological Ridge baseline → Qlib LGBModel → walk-forward 승격 gate